In [ ]:
sim = '1001'  # current steps for firing curve control
# sim = '1002'  # current steps for firing curve 20% KCNQ

# sim = '1003'  # current steps for firing curve control exact rheobase
# sim = '1004'  # current steps for firing curve 20% KCNQ exact rheobase

import sys
print("Python version:", sys.version)

# # Neuron
from neuron import h
print("NEURON executable:", h.nrnversion())

In [ ]:
############################################## SETTINGS ##############################################  
# settings
cell_type='dspn'
# cell_type='ispn'

model = 3
current_step = False

# import default settings
# import default settings
import os, sys

cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

os.chdir(main_dir)
sys.path.insert(0, main_dir)

%run -i settings.py

record_mechs = True
record_currents = True

current_step = True
if current_step:
    sim_time = 2500 
    step_end = sim_time - 200
    step_duration = 2000
    step_start = step_end - step_duration
    step_current = -150 # +128 is rheobase under control
else:
    sim_time = 500
    step_current = None
    step_duration = None

showplot = True 
rel_glut_onsets=None

mechs2record = ['kaf', 'kas', 'kir', 'kcnq']

if sim in ['1003', '1004']:
    if cell_type == 'dspn':
        current_step_range = list(range(153, 158, 1))  # ~155 pA    
    elif cell_type == 'ispn':
        current_step_range = list(range(128, 133, 1))  # ~129 pA
else:
    current_step_range = list(range(80, 401, 10))  

print(step_start, step_duration, step_end)

if sim in ['1002', '1004']:
    # scale_factors = {'kir': 0.8, 'kcnq': 0.2}
    scale_factors = {'kcnq': 0.2}

dt = 0.025
save = True

In [ ]:
# select synapses that are > 150 um from soma
from analysis_functions import *

dend2remove1 = ['dend[55]', 'dend[19]', 'dend[42]']
dend2remove1 = None

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove1)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width = 600
height = 600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()

In [ ]:
############################################### sim 1001 ###############################################   
# various current steps
# gaba and glut off
sims100x = ['1001', '1002', '1003', '1004', '1005', '1006', '1007', '1008']

# Checking if sim is in the combined list

if sim in sims100x:
    
    # (nearest synapse to relative location is selected)
    axoshaft = False # if True then places glutamatergic synapse on the shaft        
    dend_glut = ['dend[15]']

    glut = False
    
    glut_frequency = 1000 # every 1ms
    num_gluts = 0
    
    glutamate_locations = None # spine_locator(cell_type=cell_type, specs=specs, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, dend_glut=dend_glut, num_gluts=num_gluts, dend2remove=dend2remove)

    record_dendrite = 'soma[0]'
    record_location = [0.5]

    # timing_range  = list(range(120, 261, 1))
    timing_range = np.repeat(160, len(current_step_range)) 
    gaba_range = np.repeat(False, len(current_step_range)) 
    glut_range = np.repeat(True, len(current_step_range)) 
    
    # each sim will produce some plots; turn off if large number of sims
    if save:
        Nsim_save = False
        Nsim_plot = False
    
    Nsims = len(timing_range)
    

In [ ]:
# Determine which list 'sim' belongs to
sim_list=None
sim_list = [s for s in sims100x if s.startswith(sim[:4])]
  
if sim_list:
    for sim in sim_list:

        # to store sim-generated variables
        data_dict = {
                'v_all_3D': {},
                'Ca_all_3D': {},
                'imp_all_3D': {},
                'i_mechs_3D': {},
                'v_dend_tree': {},
                'v_spine_tree': {},
                'v_branch': {},
                'vsoma': [],
                'vdend': [],
                'v_dend_activated': {},
                'vspine': [],
                'v_spine_activated': {},
                'Ca_soma': [],
                'Ca_dend': [],
                'Ca_spine': [],
                'timing': [],
                'dists': [],
                'dists_spine': [],
                'i_mechs_dend': {},
                'i_mechs_dend_tree': {},
                'i_mechs_spine_tree': {},
                'i_ampa': {},
                'i_nmda': {},
                'i_gaba': {},
                'g_gaba': {},
                'record_dists':[],
                'record_spine':[],
                'spine_dist': [],
                'capacitance': [],
                'timestamp': {}
                }

        # get coordinates for all unique segments within all sections
        cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)
        mechs=['pas', 'kdr', 'naf', 'kaf', 'kas', 'kir', 'cal12', 'cal13', 'can', 'car', 'cav32', 'cav33', 'sk', 'bk']
        mech_distr_3D = record_mechs_distr_3D(cell=cell, mechs=mechs)

        # set these to False is they are not already assigned
        glut_reversal = globals().get('glut_reversal', 0) 
        vary_location = globals().get('vary_location', False)  
        dend_glut_range = globals().get('dend_glut_range', None)  
        vary_gaba_location = globals().get('vary_gaba_location', False)   

        # rec_all_path = globals().get('rec_all_path', False)  # default is False
        # if False records all v from spine / dendrite to soma
        # if True records at all unique sites including those distal to the synapse

        # determine if axoshaft or axospinous glutamatergic synapse
        axoshaft = globals().get('axoshaft', False)
        axospine = not axoshaft

        if model == 1 and not vary_location:
            glutamate_locations = sorted(glutamate_locations, reverse=True)

        metadata = {}
        # Add fixed variables to metadata
        for name in variable_names:
            try:
                if name not in metadata:
                    metadata[name] = eval(name)
            except NameError:
        #         print(f"Variable {name} not found!") # no need to print not found variables
                continue

        # syn_reversals(cell_type, specs, spine_per_length, frequency, d_lambda, dend_glut, glut_reversal, glutamate_locations, num_gluts, dend_gaba, gaba_reversal, gaba_locations, num_gabas, sim_time, dt=0.025, v_init=-84)


        # Common parameters
        common_params = {
            'cell_type': cell_type, 
            'specs': specs, 
            'spine_per_length':spine_per_length,
            'soma_diameter':soma_diameter,
            'frequency': frequency,
            'd_lambda': d_lambda,
            'glut_reversal': glut_reversal,
            'num_gluts': num_gluts,
            'gaba_reversal': gaba_reversal,
            'num_gabas': num_gabas,
            'sim_time': stim_time,
            'dt': dt,
            'v_init': v_init,
            'dend2remove': dend2remove,
        }

        if not vary_location:
            syn_reversals_params = {
                'dend_glut': dend_glut,
                'glutamate_locations': glutamate_locations,
                'dend_gaba': dend_gaba,
                'gaba_locations': gaba_locations,
                **common_params
            }
            glut_reversals, gaba_reversals = syn_reversals(**syn_reversals_params)

        else:
            syn_reversals_params = {
                **common_params
            }
            if vary_gaba_location:
                syn_reversals_params.update({
                    'dend_glut': dend_glut,
                    'glutamate_locations': glut_locations,
                    'dend_gaba': dend_gaba_range,
                    'gaba_locations': gaba_locations_range,
                })
                glut_reversals, gaba_reversals_range = syn_reversals(**syn_reversals_params)
                gaba_reversals_range = gaba_reversals_range * len(gaba_locations_range)
            else:
                syn_reversals_params.update({
                    'dend_glut': dend_glut_range,
                    'glutamate_locations': glut_locations_range,
                    'dend_gaba': dend_gaba,
                    'gaba_locations': gaba_locations,
                })
                glut_reversals_range, gaba_reversals = syn_reversals(**syn_reversals_params)
                glut_reversals_range = glut_reversals_range * len(glut_locations_range)


        # perform Nsim simulations 
        for ii in tqdm(range(Nsims)): # for ii in tqdm(list(range(0,6))):
            step_current = current_step_range[ii]
            
            # time stamp date
            time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

            sim_image_path = None
            if Nsim_save:
                sim_image_path = 'downsample/{}/model{}/{}/images/sim{}/{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim, ii)
                os.makedirs(sim_image_path, exist_ok=True)

            # set variables for this simulation
            timing = timing_range[ii]
            gaba = gaba_range[ii]
            glut = glut_range[ii]
            if vary_gaba_time:
                glut_time = stim_time
                gaba_time = timing
            else:
                glut_time = timing
                gaba_time = stim_time
            if vary_location:
                if vary_gaba_location: 
                    dend_gaba = [dend_gaba_range[ii]]
                    gaba_reversals = [gaba_reversals_range[ii]] if num_gabas == 1 else gaba_reversals_range[ii]

                else:
                    dend_glut = [dend_glut_range[ii]]
                    glut_reversals = [glut_reversals_range[ii]] if num_gluts == 1 else glut_reversals_range[ii]

                record_dendrite = record_dends[ii]
                glutamate_locations = [glut_locations_range[ii]]
                gaba_locations = [gaba_locations_range[ii]]
                record_location = [record_locs[ii]]

            if tonic:
                gbar_gaba = gbar_gaba_range[ii]
                print('tonic GABA ', gbar_gaba)
            else:
                gbar_gaba = 0

            protocol = '{}'.format(ii)

            # Run sim 
            i_recording_site, v_recording_site, v_all_3D, Ca_all_3D, imp_all_3D, i_mechs_3D, vspine, v_spine_activated, vdend, \
            v_dend_activated, vsoma, v_dend_tree, v_spine_tree, Ca_spine, Ca_dend, Ca_soma, \
            i_mechs_dend, i_mechs_dend_tree, i_mechs_spine_tree, v_branch, zdend, ztransfer, \
            ampa_currents, nmda_currents, gaba_currents, gaba_conductances, record_dist, \
            record_spine, spine_dist, cap, dt, burn_time, start_time = \
            SimClamp(sim_time = sim_time, 
                     stim_time = stim_time,
                     baseline = baseline,
                     glut = glut, 
                     glut_frequency = glut_frequency, 
                     glutamate_locations = glutamate_locations, 
                     glut_reversals = glut_reversals,
                     glut_time = glut_time, 
                     num_gluts = num_gluts, 
                     dend_glut = dend_glut, 
                     g_AMPA = g_AMPA,
                     ratio = ratio,
                     gaba = gaba, 
                     gaba_frequency = gaba_frequency, 
                     gaba_locations = gaba_locations, 
                     gaba_reversals = gaba_reversals,
                     gaba_time = gaba_time, 
                     g_GABA = g_GABA, 
                     num_gabas = num_gabas, 
                     dend_gaba = dend_gaba, 
                     specs = specs, 
                     scale_factors = scale_factors, 
                     gaba_tau1 = gaba_tau1,
                     gaba_tau2 = gaba_tau2,
                     rel_gaba_onsets = rel_gaba_onsets, 
                     rel_glut_onsets = rel_glut_onsets,
                     frequency = frequency,
                     d_lambda = d_lambda,
                     dend2remove = dend2remove,
                     v_init = v_init,
                     AMPA = AMPA,
                     NMDA = NMDA,
                     physiological = physiological,
                     timing_range = timing_range,
                     add_noise = add_noise,
                     beta = beta,                   
                     B = B,                      
                     add_sine = add_sine, 
                     axoshaft = axoshaft,
                     cell_type = cell_type,
                     current_step = current_step,
                     step_current = step_current,
                     step_duration = step_duration,
                     step_start = step_start,
                     holding_current = holding_current,
                     add_ramp = add_ramp,
                     ramp_amplitude = ramp_amplitude,
                     Cm = Cm,
                     Ra = Ra,
                     spine_per_length = spine_per_length,
                     spine_neck_diam = spine_neck_diam,
                     spine_neck_len = spine_neck_len,
                     spine_head_diam = spine_head_diam,
                     soma_diameter = soma_diameter,
                     space_clamp = space_clamp,
                     record_dendrite = record_dendrite, 
                     record_location = record_location, 
                     record_currents = record_currents,
                     record_branch = record_branch,
                     dend_glut2 = dend_glut2, 
                     record_mechs = record_mechs,
                     mechs2record = mechs2record,
                     record_path_dend = record_path_dend,   
                     record_path_spines = record_path_spines,  
                     record_all_path = record_all_path,
                     record_3D = record_3D,         
                     record_3D_impedance = record_3D_impedance,
                     freq = freq,                   
                     record_3D_mechs = record_3D_mechs,    
                     record_Ca = record_Ca,
                     record_3D_Ca = record_3D_Ca,
                     tonic = tonic,               
                     gbar_gaba = gbar_gaba,
                     rectification = rectification,       
                     distributed = distributed,         
                     gaba_params = gaba_params,
                     tonic_gaba_reversal = tonic_gaba_reversal,
                     dt = dt
            )

            # only do if want to view each sim or save sim graphs
            cols = palette_cols('brbg', Nsims)
            if Nsim_plot or Nsim_save:
                x = np.arange(0,len(vsoma)*dt, dt)
                fig = plot9(x=x,ydict=[vsoma],xaxis_range=[step_start-100, sim_time], yaxis_range=[-110,40], col=cols[ii])
                fig.update_layout(font=dict(family='Calibri', size=10, color='black'))
                fig.update_traces(line_simplify=False)  # keep full resolution
                if Nsim_plot:
                    fig.show()
                if Nsim_save:
                    fig.write_image(f"{sim_image_path}/fig_step.svg", format='svg', scale=3)

            update_data_dictionary(data_dict=data_dict, protocol=protocol, v_all_3D=v_all_3D, 
                    Ca_all_3D=Ca_all_3D, imp_all_3D=imp_all_3D, i_mechs_3D=i_mechs_3D, 
                    vspine=vspine, v_spine_activated=v_spine_activated, vdend=vdend, v_dend_activated=v_dend_activated, 
                    vsoma=vsoma, v_dend_tree=v_dend_tree, v_spine_tree=v_spine_tree, Ca_spine=Ca_spine, 
                    Ca_dend=Ca_dend, Ca_soma=Ca_soma, timing=timing, i_mechs_dend=i_mechs_dend, 
                    i_mechs_dend_tree=i_mechs_dend_tree, i_mechs_spine_tree=i_mechs_spine_tree, v_branch=v_branch, 
                    ampa_currents=ampa_currents, nmda_currents=nmda_currents, gaba_currents=gaba_currents, 
                    gaba_conductances=gaba_conductances, record_dist=record_dist, time=time, 
                    record_currents=record_currents, record_spine=record_spine, spine_dist=spine_dist, cap=cap)

        data_dict['metadata'] = metadata
        data_dict['metadata']['dt'] = dt
        data_dict['mechanisms_3D_distribution'] = mech_distr_3D

        data_dict['metadata']['record_location'] = record_location
        data_dict['metadata']['glutamate_locations'] = glutamate_locations

        # Save
        if save:
            simulations_path = 'downsample/{}/model{}/{}/simulations/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim)
            os.makedirs(simulations_path, exist_ok=True)
            names = list(data_dict.keys())
            for name in names:
                with open('{}/{}.pickle'.format(simulations_path, name), 'wb') as handle:
                    pickle.dump(data_dict[name], handle, protocol=pickle.HIGHEST_PROTOCOL)  
        

In [ ]:
save = True

# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)

fig = plot9(x=x,ydict=data_dict['vsoma'], _range=current_step_range, xaxis_range=[step_start-100, sim_time], yaxis_range=[-110,40])
fig.update_layout(font=dict(family='Calibri', size=10, color='black'))
fig.update_traces(line_simplify=False)  # keep full resolution

fig.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim)
    os.makedirs(sim_image_path, exist_ok=True)
    fig.write_image(f"{sim_image_path}/fig_summary.svg", format='svg', scale=3)


In [ ]:
_range_subset=[3, 9, 12]
fig = plot9(x=x,ydict=data_dict['vsoma'], _range=current_step_range, _range_subset=_range_subset, xaxis_range=[step_start-100, sim_time], yaxis_range=[-110,40])
fig.update_layout(font=dict(family='Calibri', size=10, color='black'))
fig.update_traces(line_simplify=False)  # keep full resolution

fig.show(config={"toImageButtonOptions": {"format": "svg"}})
            
if save:
    sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim)
    os.makedirs(sim_image_path, exist_ok=True)
    fig.write_image(f"{sim_image_path}/fig_summary2.svg", format='svg', scale=3)
    

In [ ]:
for mech in mechs2record:
    plot_mech_current(mech=mech, data_dict=data_dict, _range=current_step_range, _range_subset=_range_subset, 
                      sim_image_path=sim_image_path, sim_time=sim_time, step_start=step_start, dt=dt, save=save)

In [ ]:
from scipy.optimize import least_squares

ydict = data_dict['vsoma']
dt_spike = deltat if 'deltat' in globals() else dt

spike_counts = [count_spikes(np.asarray(trace).squeeze(), dt=dt_spike) for trace in ydict]

I = np.asarray(current_step_range, float)
spikes = np.asarray(spike_counts, float)
rate = spikes / float(step_duration)
def mod(p, x):
    I0, rmax, I50, n = p
    z = np.clip(x - I0, 0, None)
    return rmax * (z**n) / (z**n + I50**n + 1e-12)

def resid(p):
    yhat = mod(p, I)
    return (yhat - rate)

I_pos = I[rate > 0]
I0_init = np.percentile(I_pos, 10) if I_pos.size else np.min(I)
rmax_init = max(rate.max(), 1.0)
I50_init = max(np.percentile(np.clip(I - I0_init, 0, None), 60), 1e-6)
n_init = 2.0
p0 = np.array([I0_init, rmax_init, I50_init, n_init])
lb = np.array([np.min(I)-50.0, 0.0, 1e-6, 0.5])
ub = np.array([np.max(I), rate.max()*10 + 100.0, (np.max(I)-np.min(I))*5 + 1000.0, 6.0])

res = least_squares(resid, p0, bounds=(lb, ub), max_nfev=2000)
p = res.x

I_fit = np.linspace(I.min(), I.max(), 300)
rate_fit = mod(p, I_fit)

lw = 1
plot_color = 'grey'

fig = go.Figure()
fig.add_trace(go.Scatter(x=I, y=spikes, mode='markers', marker=dict(size=6, color=plot_color), name='Spikes'))
fig.add_trace(go.Scatter(x=I_fit, y=rate_fit*step_duration, mode='lines', line=dict(color=plot_color, width=lw), name='Hill fit'))
fig.update_layout(
    title=dict(
        text='Spike Count vs Current Step',
        x=0.5,
        xanchor='center',
        font=dict(color=plot_color, size=12)
    ),
    xaxis=dict(title='Current Step (pA)', 
               showgrid=False, 
               zeroline=False, 
               ticks='outside',
               showline=True, 
               mirror=False, 
               linecolor=plot_color,
               tickcolor=plot_color,
               linewidth=lw, 
               tickwidth=lw),
    yaxis=dict(title='Spike Count', 
               showgrid=False,
               zeroline=False, 
               ticks='outside',
               showline=True, 
               mirror=False, 
               linecolor=plot_color,
               tickcolor=plot_color,
               linewidth=lw, 
               tickwidth=lw,
               range=[-5, 100]),
    template='plotly_white', 
    font=dict(family='Calibri', size=10, color=plot_color),
    width=500, 
    height=400,
    paper_bgcolor='rgba(0,0,0,0)', 
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.update_traces(line_simplify=False)

fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    fig.write_image(f"{sim_image_path}/fig_spike_count_vs_steps_summary.svg", format='svg', scale=3)        

In [ ]:
# pull out somatic conductances from json
import json

if cell_type == 'dspn':
    params='params_dMSN3.json'
elif cell_type == 'ispn':
    params='params_iMSN3.json'     

with open(params, 'r') as f:
    params_dict = json.load(f)

gbar = {}
for k in ['kaf', 'kas', 'kir', 'kcnq', 'pas']:
    key = f"gbar_{k}_somatic" if k != 'pas' else 'g_pas_all'
    val = float(params_dict[key]['Value'])
    sf = scale_factors.get(k, 1) if 'scale_factors' in locals() and isinstance(scale_factors, dict) else 1
    gbar[k] = val * sf

def kaf_g(v, gbar):
    a_m = 1.5 / (1 + np.exp((v - 5) / -17))
    b_m = 0.6 / (1 + np.exp((v - 11) / 9))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.105 / (1 + np.exp((v + 120) / 22))
    b_h = 0.065 / (1 + np.exp((v + 54) / -11))
    h_inf = a_h / (a_h + b_h)
    return gbar * (m_inf**2) * h_inf

def kas_g(v, gbar):
    a_m = 0.25 / (1 + np.exp((v - 50) / -20))
    b_m = 0.05 / (1 + np.exp((v + 90) / 35))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.0025 / (1 + np.exp((v + 95) / 16))
    b_h = 0.002 / (1 + np.exp((v - 50) / -70))
    h_inf = 0.2 + (a_h / (a_h + b_h)) * 0.8
    return gbar * (m_inf**2) * h_inf

def kir_g(v, gbar):
    m_inf = 1 / (1 + np.exp((v + 102) / 13))
    return gbar * m_inf

def kcnq_g(v, gbar):
    m_inf = 1 / (1 + np.exp((-59.5 - v) / 10.3))
    return gbar * (m_inf**4)

v_rest = -85
v_test = -60
Cm = 180e-12
funcs = {'kaf': kaf_g, 'kas': kas_g, 'kir': kir_g, 'kcnq': kcnq_g}

g_rest = {k: f(v_rest, gbar[k]) for k, f in funcs.items()}
g_rest['pas'] = gbar['pas']
total_g_rest = sum(g_rest.values())
Rin_rest = 1 / (total_g_rest * 1e-4) / 1e6

g_60 = {k: f(v_test, gbar[k]) for k, f in funcs.items()}
g_60['pas'] = gbar['pas']
total_g_60 = sum(g_60.values())
Rin_60 = 1 / (total_g_60 * 1e-4) / 1e6

v_range = np.linspace(-90, -40, 200)
Rin_vals = []
for v in v_range:
    g_tot = sum([f(v, gbar[k]) for k, f in funcs.items()]) + gbar['pas']
    Rin_vals.append(1 / (g_tot * 1e-4) / 1e6)

V_th = -59.5
ΔV_rest = V_th - v_rest
lo, hi = sorted([v_rest, V_th])
mask = (v_range >= lo) & (v_range <= hi)
Rin_path = np.array(Rin_vals)[mask]
Rin_peak = float(np.max(Rin_path))
Rin_at_th = float(np.interp(V_th, v_range, Rin_vals))

plot_color = 'gray'

layout_common = dict(
    width=560, height=340,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=160, t=20, b=50)
)

lw = 1
ymin = min(Rin_vals)
ymax = max(Rin_vals)
xrange_common = [-100, -40]

# Fig 1 Input resistance vs Vm
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=v_range, y=Rin_vals,
    mode='lines',
    line=dict(color=plot_color, width=lw),
    name='Rin'
))
fig1.add_trace(go.Scatter(
    x=[V_th, V_th],
    y=[ymin, ymax],
    mode='lines',
    line=dict(color=plot_color, dash='dot', width=lw),
    name='V_th'
))
fig1.update_layout(
    **layout_common,
    xaxis=dict(title='Membrane potential (mV)', range=xrange_common,
               showgrid=False, ticks='outside',
               mirror=False, linecolor=plot_color, tickcolor=plot_color,
               showline=True, zeroline=False),
    yaxis=dict(title='Input resistance (MΩ)', range=[ymin, ymax],
               showgrid=False, ticks='outside',
               mirror=False, linecolor=plot_color, tickcolor=plot_color,
               showline=True, zeroline=False),
    showlegend=False,
    font=dict(family='Calibri', size=10, color=plot_color)
)
fig1.update_traces(line_simplify=False)
fig1.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig1.write_image(f"{sim_image_path}/fig_input_resistance_vs_Vm_summary.svg", format='svg', scale=3)

# Fig 2 Conductance vs Vm (semilog)
cols = palette_cols('berlin', len(funcs), alpha=0.8)
semilog = True
fig2 = go.Figure()
for (name, func), col in zip(funcs.items(), cols):
    fig2.add_trace(go.Scatter(
        x=v_range, y=func(v_range, gbar[name]),
        mode='lines',
        line=dict(color=col, width=lw),
        name=name
    ))

xaxis_style = dict(
    title='Membrane potential (mV)',
    range=xrange_common,
    showgrid=False,
    ticks='outside',
    mirror=False,
    linecolor=plot_color,
    tickcolor=plot_color,
    showline=True,
    zeroline=False
)
yaxis_style = dict(
    title='Conductance (S/cm²)',
    showgrid=False,
    ticks='outside',
    mirror=False,
    linecolor=plot_color,
    tickcolor=plot_color,
    showline=True,
    zeroline=False
)
if semilog:
    major_ticks = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0]
    minor_ticks = []
    for i in range(len(major_ticks) - 1):
        base = major_ticks[i]
        minor_ticks += [base * f for f in range(2, 10)]
    yaxis_style.update(
        type='log',
        tickvals=major_ticks,
        ticktext=[f"10<sup>{int(np.log10(val))}</sup>" for val in major_ticks],
        minor=dict(tickvals=minor_ticks, ticks='outside', showgrid=False),
        showexponent='none'
    )
fig2.update_layout(
    **layout_common,
    xaxis=xaxis_style,
    yaxis=yaxis_style,
    showlegend=True,
    legend=dict(borderwidth=0, x=1.05, y=1, xanchor='left', yanchor='top',
                font=dict(color=plot_color)),
    font=dict(family='Calibri', size=10, color=plot_color)
)

fig2.update_traces(line_simplify=False)
fig2.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig2.write_image(f"{sim_image_path}/fig_conductance_vs_Vm_summary.svg", format='svg', scale=3)

If linear-scale plot looks exponential on a semilog axis (i.e. straight line on log scale), that means the conductance changes exponentially with voltage usually indicating strong voltage dependence, which can correspond to either inward or outward rectification depending on direction.



In [ ]:
v_neg = -100
v_pos = -40

rectification = {}
for name, func in funcs.items():
    g_neg = func(v_neg, gbar[name])
    g_pos = func(v_pos, gbar[name])
    ratio = g_neg / g_pos if g_pos != 0 else np.nan
    if ratio > 1.2:
        rect_type = 'inward rectifier'
    elif ratio < 0.8:
        rect_type = 'outward rectifier'
    else:
        rect_type = 'linear (non-rectifying)'
    rectification[name] = (ratio, rect_type)

print('Rectification indices:')
for name, (ratio, rect_type) in rectification.items():
    print(f"{name:5s}: G(-100)/G(-40) = {ratio:.2f} : {rect_type}")